# Reusable Rayleigh gravity response workflow

Build the frequency response of a homogeneous Rayleigh half-space once, then apply it to multiple vertical displacement phasor spectra. The outputs are first-order field changes at fixed observation points. Their linearization applicability has not been assessed; this workflow makes no finite-motion accuracy claim. Complex values are linear amplitudes, not time-domain fields.

In [ ]:
import os
import sys
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u
from gwexpy.frequencyseries import FrequencySeries
from IPython.display import JSON, display

import gragra
from gragra.analytic.seismic import rayleigh_halfspace, rayleigh_response
from gragra.geometry import Frame, HalfSpace
from gragra.results import PhasorRepresentation, compare
from gragra.targets import TargetPoints

repository = Path.cwd().resolve()
expected_gragra = (repository / "src" / "gragra").resolve()
imported_gragra = Path(gragra.__file__).resolve().parent
if imported_gragra != expected_gragra:
    raise RuntimeError(f"expected gragra from {expected_gragra}, got {imported_gragra}")
provenance = {
    "python_executable": str(Path(sys.executable).resolve()),
    "python_prefix": str(Path(sys.prefix).resolve()),
    "gragra_import_path": str(imported_gragra),
    "gragra_version": version("gragra"),
    "gwexpy_version": version("gwexpy"),
}
display(JSON(provenance))

## Define the half-space and targets

The surface frame defines the free surface at local `z=0`, with the medium at `z<0`. Frequencies are irregularly spaced and expressed in hertz. The response input is vertical displacement at the stated phase origin, in metres.

In [ ]:
surface = Frame(name="surface")
halfspace = HalfSpace(surface)
rho = 2300 * u.kg / u.m**3
vp = 5800 * u.m / u.s
vs = 3200 * u.m / u.s
frequencies = np.array([0.8, 2.1, 4.7]) * u.Hz
phase_origin = np.array([-4.0, 9.0, 0.0]) * u.m
direction = np.array([1.0, 2.0, 0.0])
targets = TargetPoints(
    np.array([[3, -1, 2], [3, -1, -2], [12, 9, 0.2], [-4, 7, -30]]) * u.m,
    ids=("above A", "below B", "above C", "below D"),
    frame=surface,
)

response_setup = rayleigh_response(
    halfspace,
    density=rho,
    vp=vp,
    vs=vs,
    frequencies=frequencies,
    direction=direction,
    phase_origin=phase_origin,
).at(targets)
print("target validation:", response_setup.validate())
response = response_setup.run()
contract = response.contract
print("response contract:")
print(f"  input: {contract.input_name} ({contract.input_quantity}), unit={contract.input_unit}")
print(f"  input location: {contract.input_location_m} in {contract.input_frame.name}")
print(f"  mapping: {contract.mapping_kind}; action={contract.action}; storage={contract.storage}")
print(f"  output: {response.output_definition.kind}; change={response.output_definition.change.form}")
print(f"  linearization assessment: {contract.linearization_assessment}")
print("  coefficients are per metre of vertical displacement:")
for name in ("potential", "acceleration", "gradient"):
    matrix = response.coefficients(name)
    print(f"    {name}: {matrix.units[0, 0]}, shape={matrix.shape}, rows={list(matrix.rows)}, columns={list(matrix.cols)}")

## Reuse the same response for two spectra

Each application supplies an explicit phasor representation and GPS reference time. The response coefficients contain neither spectrum, so both applications reuse the same precomputed transfer.

In [ ]:
displacement_a = FrequencySeries(
    np.array([1.0 + 0.2j, 0.6 - 0.4j, -0.3 + 0.7j]) * u.nm,
    frequencies=frequencies,
    name="spectrum A vertical displacement",
)
displacement_b = FrequencySeries(
    np.array([-0.5 + 0.8j, 1.2 + 0.1j, 0.4 - 0.6j]) * u.nm,
    frequencies=frequencies,
    name="spectrum B vertical displacement",
)
representation = PhasorRepresentation()
result_a = response.apply(
    vertical_displacement=displacement_a,
    representation=representation,
    reference_time=(1_400_000_000, 0),
)
result_b = response.apply(
    vertical_displacement=displacement_b,
    representation=representation,
    reference_time=(1_500_000_000, 250_000_000),
)
print("A reference time:", result_a.context.reference_time_gps)
print("B reference time:", result_b.context.reference_time_gps)
applied_evaluation = result_a.evaluation
assert result_a.context is applied_evaluation.context
print("applied evaluation context shares result context:", result_a.context is applied_evaluation.context)
for label, result in (("A", result_a), ("B", result_b)):
    binding = result.context.binding("acceleration")
    assert binding is result.bindings["acceleration"]
    matrix = result.acceleration
    print(f"spectrum {label} acceleration: {matrix.units[0, 0]}; rows={list(matrix.rows)}; columns={list(matrix.cols)}")
display(result_a.acceleration)
display(result_b.acceleration)

## Check one application against direct evaluation

The applied result for spectrum A is compared with the direct Rayleigh half-space calculation under the same physical inputs and reference time. This checks that applying the stored response reproduces the direct calculation for this example.

In [ ]:
direct_a = rayleigh_halfspace(
    halfspace,
    rho,
    vp,
    vs,
    displacement_a,
    representation,
    (1_400_000_000, 0),
    phase_origin,
    direction,
).at(targets).run()
comparison = compare(reference=direct_a, candidate=result_a)
display(comparison.summary)
print("component phase differences:")
display(comparison.phase_differences)

## Plot a selected response

The plot shows the amplitude of the vertical acceleration component for two targets. Values remain complex phasor amplitudes under the declared convention; they are not reconstructed time-domain fields.

In [ ]:
matrix = result_a.acceleration
rows = list(matrix.rows)
columns = list(matrix.cols)
vertical_index = next(i for i, label in enumerate(columns) if str(label).lower() in {"z", "vertical"})
values = np.asarray(matrix.value)
frequency_hz = matrix.frequencies.to_value(u.Hz)
selected = ("above A", "below B")
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for target in selected:
    row = rows.index(target)
    ax.plot(frequency_hz, np.abs(values[row, vertical_index, :]), marker="o", label=target)
ax.set(xlabel="Frequency (Hz)", ylabel=f"Vertical acceleration amplitude ({matrix.units[0, 0]})")
ax.set_title("Spectrum A: vertical acceleration phasor amplitude")
ax.grid(True, alpha=0.3)
ax.legend(title="Target")
fig.tight_layout()
plot_path = Path(os.environ.get(
    "GRAGRA_RAYLEIGH_RESPONSE_PLOT",
    Path.home() / ".cache" / "gragra" / "rayleigh-response-workflow" / "rayleigh_response_workflow.png",
)).expanduser().resolve()
plot_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(plot_path, dpi=160, bbox_inches="tight")
plt.show()
print("plot:", plot_path)